# From SQL to Power BI: Preparing Data with Python
##### Professor: Joanna
##### Developed by: Ariana Ghimire

<hr style="border: 2px solid #003262">

<hr style="border: 2px solid #C9B676">

## 1. Introduction

You already know how to **query** data with SQL. In real analytics jobs, that is only the start.

After data is collected and queried, teams often:

1. Clean and reshape it in **Python** (pandas / Polars)
2. Export a tidy CSV
3. Build an interactive **Power BI** dashboard

This notebook teaches that middle step — preparing Spotify song data for Power BI.

```
SQL Database
      ↓
SQL Queries
      ↓
Python (pandas / Polars)
      ↓
Cleaned Dataset (CSV)
      ↓
Power BI Dashboard
```

> **Big idea:** SQL retrieves data. Python prepares it. Power BI presents it.

## Learning Objectives

By the end of this notebook, you should be able to:

- Load and inspect a CSV with pandas
- Clean columns, missing values, and dates
- Translate SQL `WHERE`, `SELECT`, `ORDER BY`, and `GROUP BY` into pandas (and Polars)
- Make a few quick exploration charts in Python
- Export a dashboard-ready CSV for Power BI
- Sketch the visuals a Power BI dashboard should include

> **How to use this notebook:** Read each short section, run the example cell, then try the practice. Expand **Solution** only after you try.

<hr style="border: 2px solid #003262">

## 2. Load Data

We will use a Spotify songs sample in `data.csv` (a smaller teaching subset of a public Spotify dataset).

Compare loading data in Python to SQL:

```sql
SELECT *
FROM songs
LIMIT 5;
```

In [ ]:
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

songs = pd.read_csv("data.csv")
songs.head()

In [ ]:
songs.info()

In [ ]:
songs.describe()

Discuss what you see:

- **Rows** — each song is one observation
- **Columns** — features like `name`, `artists`, `year`, `popularity`
- **Data types** — numbers, text, dates (as strings until we convert them)

`info()` is like checking column types. `describe()` summarizes numeric columns.

**Practice:** Print the number of rows and columns using `.shape`, then list the column names.

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
print(songs.shape)
print(list(songs.columns))
```

</details>

In [ ]:
# YOUR CODE HERE


<hr style="border: 2px solid #003262">

## 3. Basic Cleaning

Power BI works best with **clean, focused** tables. Common preprocessing steps:

### Remove unnecessary columns

The `id` column is useful in a database, but not needed for our dashboard.

In [ ]:
songs = songs.drop(columns=["id"])
songs.head(2)

### Missing values

Check how many nulls each column has:

In [ ]:
songs.isna().sum()

### Convert dates

`release_date` is currently text. Convert it so year/decade work reliably.

In [ ]:
songs["release_date"] = pd.to_datetime(songs["release_date"], errors="coerce")
songs[["name", "release_date"]].head()

### Create a decade column

This is something students immediately appreciate in Power BI — decades make great bar charts.

In [ ]:
songs["decade"] = (songs["year"] // 10) * 10
songs[["name", "year", "decade"]].head()

**Practice:** Create a new column `duration_min` that converts `duration_ms` from milliseconds to minutes (divide by 60000). Show `name` and `duration_min` for the first 5 rows.

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
songs["duration_min"] = songs["duration_ms"] / 60000
songs[["name", "duration_min"]].head()
```

</details>

In [ ]:
# YOUR CODE HERE


<hr style="border: 2px solid #003262">

## 4. Filtering (SQL → pandas → Polars)

**SQL**
```sql
SELECT *
FROM songs
WHERE popularity > 60;
```

**pandas**

In [ ]:
popular = songs[songs["popularity"] > 60]
popular[["name", "artists", "popularity"]].head()

**Polars**

In [ ]:
songs_pl = pl.from_pandas(songs)
songs_pl.filter(pl.col("popularity") > 60).select("name", "artists", "popularity").head()

**Practice:** Filter songs where `year >= 2010` and show `name`, `year`, and `popularity` (first 5 rows).

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
songs[songs["year"] >= 2010][["name", "year", "popularity"]].head()
```

</details>

In [ ]:
# YOUR CODE HERE


<hr style="border: 2px solid #003262">

## 5. Selecting Columns

**SQL**
```sql
SELECT name, artists, popularity
FROM songs;
```

**pandas**

In [ ]:
songs[["name", "artists", "popularity"]].head()

**Polars**

In [ ]:
songs_pl.select("name", "artists", "popularity").head()

**Practice:** Select only `name`, `decade`, `energy`, and `danceability`. Show the first 5 rows.

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
songs[["name", "decade", "energy", "danceability"]].head()
```

</details>

In [ ]:
# YOUR CODE HERE


<hr style="border: 2px solid #003262">

## 6. Sorting

**SQL**
```sql
SELECT name, popularity
FROM songs
ORDER BY popularity DESC;
```

**pandas**

In [ ]:
songs.sort_values("popularity", ascending=False)[["name", "artists", "popularity"]].head(10)

**Polars**

In [ ]:
songs_pl.sort("popularity", descending=True).select("name", "artists", "popularity").head(10)

**Practice:** Sort by `energy` descending and show the top 5 song names and energy values.

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
songs.sort_values("energy", ascending=False)[["name", "energy"]].head(5)
```

</details>

In [ ]:
# YOUR CODE HERE


<hr style="border: 2px solid #003262">

## 7. GROUP BY (Perfect for Power BI)

Aggregations you build in Python often become the same summaries Power BI charts display.

**SQL**
```sql
SELECT
  year,
  AVG(popularity) AS avg_popularity
FROM songs
GROUP BY year;
```

**pandas**

In [ ]:
songs.groupby("year")["popularity"].mean().head(10)

Average popularity by **decade** (great Power BI bar chart):

In [ ]:
by_decade = songs.groupby("decade")["popularity"].mean().reset_index(name="avg_popularity")
by_decade

**Polars**

In [ ]:
songs_pl.group_by("decade").agg(
    pl.col("popularity").mean().alias("avg_popularity")
).sort("decade")

**Practice:** Group by `decade` and compute the average `danceability`. Sort by decade ascending.

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
songs.groupby("decade")["danceability"].mean().reset_index(name="avg_danceability").sort_values("decade")
```

</details>

In [ ]:
# YOUR CODE HERE


<hr style="border: 2px solid #003262">

## 8. Simple Visualizations in Python

Python is great for **quick exploration**. Power BI is designed for **interactive dashboards**.

Make only a few charts here — just enough to understand the data before exporting.

### Average popularity by year

In [ ]:
avg_by_year = songs.groupby("year")["popularity"].mean()
avg_by_year.plot(figsize=(10, 4), title="Average Popularity by Year")
plt.xlabel("Year")
plt.ylabel("Average Popularity")
plt.tight_layout()
plt.show()

### Average popularity by decade (bar chart)

In [ ]:
by_decade.plot(x="decade", y="avg_popularity", kind="bar", legend=False, figsize=(8, 4),
              title="Average Popularity by Decade")
plt.xlabel("Decade")
plt.ylabel("Average Popularity")
plt.tight_layout()
plt.show()

### Popularity distribution (histogram)

In [ ]:
songs["popularity"].plot(kind="hist", bins=20, figsize=(8, 4), title="Popularity Distribution")
plt.xlabel("Popularity")
plt.tight_layout()
plt.show()

### Top artists by number of songs

In [ ]:
top_artists = songs["artists"].value_counts().head(10)
top_artists.plot(kind="barh", figsize=(8, 5), title="Top 10 Artists by Song Count")
plt.xlabel("Number of Songs")
plt.tight_layout()
plt.show()

> **Remember:** These charts are for exploration. The polished interactive version belongs in Power BI.

**Try It Yourself:** Make a line chart of average `energy` by `year`.

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
songs.groupby("year")["energy"].mean().plot(figsize=(10, 4), title="Average Energy by Year")
plt.xlabel("Year")
plt.ylabel("Average Energy")
plt.tight_layout()
plt.show()
```

</details>

In [ ]:
# YOUR CODE HERE


<hr style="border: 2px solid #003262">

## 9. Prepare Data for Power BI

Create a smaller cleaned dataset with only the columns the dashboard needs.

In [ ]:
dashboard = songs[
    [
        "name",
        "artists",
        "year",
        "decade",
        "popularity",
        "danceability",
        "energy",
        "valence",
        "tempo",
        "explicit",
    ]
].copy()

dashboard.head()

### Export the CSV Power BI will import

In [ ]:
dashboard.to_csv("spotify_dashboard.csv", index=False)
print("Exported spotify_dashboard.csv with", len(dashboard), "rows")

**Practice:** Confirm the export worked by reading `spotify_dashboard.csv` back into a DataFrame called `check` and printing `check.shape`.

*Write your answer in the cell below.*

<details>
<summary><strong>Solution:</strong></summary>

```python
check = pd.read_csv("spotify_dashboard.csv")
print(check.shape)
```

</details>

In [ ]:
# YOUR CODE HERE


<hr style="border: 2px solid #003262">

## 10. Open Power BI

Very short setup:

1. Open **Power BI Desktop**
2. Choose **Get data → Text/CSV**
3. Select `spotify_dashboard.csv` from this `PowerBI` folder
4. Click **Load**

You should see columns like `name`, `artists`, `year`, `decade`, `popularity`, `danceability`, `energy`, `valence`, `tempo`, and `explicit`.

<hr style="border: 2px solid #003262">

## 11. Dashboard Challenge

Recreate these visuals in Power BI using `spotify_dashboard.csv`.

### Visual 1 — Average popularity by decade
- Chart type: **Bar chart**
- Axis: `decade`
- Values: Average of `popularity`

### Visual 2 — Songs released each year
- Chart type: **Line chart**
- Axis: `year`
- Values: Count of `name` (or Count of rows)

### Visual 3 — Popularity vs Energy
- Chart type: **Scatter chart**
- X axis: `energy`
- Y axis: `popularity`

### Visual 4 — Average Danceability by Decade
- Chart type: **Column chart**
- Axis: `decade`
- Values: Average of `danceability`

### Visual 5 — Slicers
Add slicers for:
- `year`
- `artists`
- `explicit`

### Visual 6 — Cards (KPIs)
Add cards for:
- Average `popularity`
- Average `energy`
- Number of songs (Count of rows)

> Arrange the page so cards are at the top, slicers on the side, and charts in the center.

<hr style="border: 2px solid #003262">

## 12. Reflection

Answer these in a Markdown cell or on paper:

1. Why clean data in Python instead of only in Power BI?
2. Which tasks were easier in Python?
3. Which tasks are easier in Power BI?
4. When would you use SQL instead of Python or Power BI?

<details>
<summary><strong>Sample answers:</strong></summary>

1. Python is better for repeatable cleaning, creating columns (like decade), and handling larger/messy transforms before reporting.
2. Bulk cleaning, creating calculated columns, quick group-bys, and exporting a tidy CSV.
3. Interactive filters/slicers, polished dashboard layout, and sharing visuals with non-coders.
4. When the data still lives in a database and you need to retrieve or join source tables efficiently.

</details>

<hr style="border: 2px solid #003262">

<hr style="border: 2px solid #C9B676">

**Workflow reminder**

`SQL → pandas/Polars → CSV → Power BI Dashboard`

*End of notebook. Use **Kernel → Restart & Run All**, then open `spotify_dashboard.csv` in Power BI.*